# Picotron 1B-token demo: DDP pretraining -> inference -> SmolTalk SFT -> inference

This Kaggle notebook exercises the nested config, reusable preprocessing tool, installed CLI, real two-rank DDP, safetensors checkpoints, and the script-first SFT trainer. The model is intentionally about 1M parameters; the 1B-token budget demonstrates the pipeline, not frontier quality.


## 1. Clone and install

Install requirements and the editable package so the picotron and picotron-sft console commands are available.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path("/kaggle/working/picotron")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/AndyMLAndAI/picotron.git", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("Installed Picotron from", REPO)


## 2. Verify the T4 hardware contract

Both GPUs must be Turing (7.5). Picotron must choose fp16, never bf16, and report attention/Triton availability.


In [ ]:
import torch
sys.path.insert(0, str(REPO / "src"))
from picotron.utils.hardware import (
    detect_attention_backend,
    detect_triton_support,
    get_gpu_compute_capability,
    select_training_dtype,
)

assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."
assert torch.cuda.device_count() >= 2, f"Expected two GPUs, found {torch.cuda.device_count()}."
for index in range(torch.cuda.device_count()):
    capability = get_gpu_compute_capability(index)
    dtype = select_training_dtype(index)
    print(index, torch.cuda.get_device_name(index), capability, dtype)
    assert capability == (7, 5)
    assert dtype is torch.float16 and dtype is not torch.bfloat16
print("attention:", detect_attention_backend(0))
print("triton (explicit opt-in only):", detect_triton_support(enabled=False, device=0))


## 3. Preprocess exactly 1B FineWeb-Edu tokens

This calls tools/preprocess_data.py; tokenization is not reimplemented here. The sample-10BT FineWeb-Edu configuration is large enough for a bounded 1B-token prefix. GPT-2's 50,257 vocabulary fits uint16, so the output is exactly 2,000,000,000 bytes. The tool intentionally concatenates documents without EOS separators.


In [ ]:
TARGET_TOKENS = 1_000_000_000
TOKEN_PATH = REPO / "data" / "fineweb_edu_1b_gpt2.uint16"
TOKEN_PATH.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, "tools/preprocess_data.py",
    "--dataset-name", "HuggingFaceFW/fineweb-edu",
    "--dataset-config", "sample-10BT",
    "--tokenizer-name", "gpt2",
    "--target-tokens", str(TARGET_TOKENS),
    "--output-path", str(TOKEN_PATH),
    "--text-field", "text",
], cwd=REPO, check=True)
assert TOKEN_PATH.stat().st_size == TARGET_TOKENS * 2
print(f"Prepared {TOKEN_PATH}; {TOKEN_PATH.stat().st_size / 2**30:.2f} GiB")


## 4. Write the nested pretraining config

Two ranks, four sequences per rank, and 256 tokens per sequence give 2,048 raw tokens per update. 488,282 updates consume exactly 1B cached input tokens because the final distributed batch is partial. The approximately 1M-parameter model uses hidden size 10 and eight small blocks; layer 3 uses NoPE and every other layer uses RoPE.


In [ ]:
%%writefile config_pretrain.yaml
checkpoints:
  checkpoint_interval: 10000
  checkpoints_path: /kaggle/working/picotron/checkpoints/pretrain_1b.safetensors
  save_final_state: true
model:
  dtype: auto
  model_config:
    vocab_size: 50257
    hidden_size: 10
    intermediate_size: 40
    num_hidden_layers: 8
    num_attention_heads: 1
    attention_type: mha
    nope_layers: [3]
    position_embedding_type: rope
    rope_theta: 10000.0
optimizer:
  learning_rate_scheduler:
    learning_rate: 0.0003
parallelism:
  dp: 2
  zero_stage: 0
tokens:
  sequence_length: 256
  micro_batch_size: 4
  # 1B / 256 = 3,906,250 examples; each rank gets 1,953,125.
  # That is 488,281 full batches plus one partial batch per rank.
  train_steps: 488282
data:
  dataset_token_path: /kaggle/working/picotron/data/fineweb_edu_1b_gpt2.uint16
  tokenizer_name: gpt2
  vocab_size: 50257
logging:
  log_level: INFO
  iteration_step_info_interval: 100
general:
  project: picotron
  run: kaggle_1b_ddp
  seed: 1337


## 5. Launch real two-rank DDP pretraining

The installed picotron console script is passed to torchrun. This is a real two-process launch, not a single-process notebook call. The CLI reads data.dataset_token_path, uses the memmap dataset with DistributedSampler, and rank 0 saves the safetensors checkpoint plus optimizer sidecar. Expect substantial runtime on two T4s.


In [ ]:
import shutil

CHECKPOINT_PATH = REPO / "checkpoints" / "pretrain_1b.safetensors"
picotron_command = shutil.which("picotron")
assert picotron_command, "pip install -e . did not install the picotron command."
subprocess.run([
    sys.executable, "-m", "torch.distributed.run",
    "--standalone", "--nproc_per_node=2",
    picotron_command, "--config", str(REPO / "config_pretrain.yaml"),
], cwd=REPO, check=True)
assert CHECKPOINT_PATH.exists()
assert CHECKPOINT_PATH.with_suffix(".optimizer.pt").exists()
print("DDP checkpoint:", CHECKPOINT_PATH)


## 6. Inference from the pretraining safetensors checkpoint

A fresh model loads the rank-0 checkpoint directly with safetensors and uses greedy generation. Output quality is intentionally limited by the tiny model.


In [ ]:
from transformers import AutoTokenizer
from safetensors.torch import load_file
from picotron.config.config import load_config
from picotron.models.picotron_decoder import PicotronDecoderModel

config = load_config(REPO / "config_pretrain.yaml")
device = torch.device("cuda:0")
tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
inference_model = PicotronDecoderModel(config).to(device)
inference_model.load_state_dict(load_file(str(CHECKPOINT_PATH), device=str(device)))
inference_model.eval()

@torch.no_grad()
def generate(model, prompt, max_new_tokens=48):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    for _ in range(max_new_tokens):
        context = ids[:, -config.tokens.sequence_length:]
        next_id = model(context)[:, -1].argmax(dim=-1, keepdim=True)
        ids = torch.cat((ids, next_id), dim=1)
    return tokenizer.decode(ids[0], skip_special_tokens=True)

pretrain_prompts = [
    "The capital city of France is",
    "The scientific reason the sky is blue is",
]
pretrain_outputs = [generate(inference_model, prompt) for prompt in pretrain_prompts]
for prompt, output in zip(pretrain_prompts, pretrain_outputs):
    print(f"PROMPT: {prompt}\n{output}\n")


## 7. Write nested SmolTalk SFT settings

The installed SFT CLI requires custom model/data factories, so this notebook uses picotron_sft's script-first run_sft API and saves its optimizer sidecar explicitly. The YAML remains a nested, reproducible record of the SFT run.


In [ ]:
%%writefile config_sft.yaml
checkpoints:
  base_checkpoint_path: /kaggle/working/picotron/checkpoints/pretrain_1b.safetensors
  output_checkpoint_path: /kaggle/working/picotron/checkpoints/smoltalk.safetensors
dataset:
  name: HuggingFaceTB/smoltalk
  config: all
  split: train
  streaming: true
  max_examples: null
tokens:
  sequence_length: 256
  micro_batch_size: 2
  train_steps: null
optimizer:
  learning_rate_scheduler:
    learning_rate: 0.0001


## 8. Stream the full SmolTalk training split

The current dataset identifier is HuggingFaceTB/smoltalk, configuration all, split train, with messages containing role/content fields. Streaming avoids a multi-gigabyte upfront download. This is full-conversation causal SFT: labels include all conversation tokens and padding is ignored; assistant-only masking is not claimed.


In [ ]:
import yaml
from datasets import load_dataset

with open(REPO / "config_sft.yaml", encoding="utf-8") as handle:
    sft_settings = yaml.safe_load(handle)
sft_tokens = sft_settings["tokens"]["sequence_length"]
sft_batch_size = sft_settings["tokens"]["micro_batch_size"]
raw_smoltalk = load_dataset(
    sft_settings["dataset"]["name"],
    sft_settings["dataset"]["config"],
    split=sft_settings["dataset"]["split"],
    streaming=True,
)

def render_messages(messages):
    if not isinstance(messages, list):
        raise ValueError("SmolTalk example has no messages list.")
    rendered = []
    for message in messages:
        if not isinstance(message, dict):
            raise ValueError("SmolTalk messages must be mappings.")
        role, content = message.get("role"), message.get("content")
        if not isinstance(role, str) or not isinstance(content, str):
            raise ValueError("SmolTalk messages require string role/content.")
        rendered.append(f"{role}: {content}")
    return "\n".join(rendered)

def collate_sft(sequences):
    longest = max(sequence.numel() for sequence in sequences)
    input_ids = torch.full(
        (len(sequences), longest), tokenizer.eos_token_id, dtype=torch.long
    )
    labels = torch.full((len(sequences), longest), -100, dtype=torch.long)
    for row, sequence in enumerate(sequences):
        input_ids[row, :sequence.numel()] = sequence
        labels[row, :sequence.numel()] = sequence
    return {"input_ids": input_ids, "labels": labels}

def smoltalk_batches():
    pending = []
    for example in raw_smoltalk:
        ids = tokenizer.encode(render_messages(example["messages"]), add_special_tokens=False)
        ids = ids[:sft_tokens]
        if len(ids) >= 2:
            pending.append(torch.tensor(ids, dtype=torch.long))
        if len(pending) == sft_batch_size:
            yield collate_sft(pending)
            pending = []
    if pending:
        yield collate_sft(pending)

print("SmolTalk stream ready; no examples were subsampled.")


## 9. Fine-tune and save the SFT safetensors checkpoint

SFT uses picotron_sft.run_sft directly so its real AdamW optimizer can be saved alongside the resulting model weights. num_steps=None consumes the complete streamed train split; runtime should be measured honestly on Kaggle.


In [ ]:
from torch.optim import AdamW
from picotron.serialize.checkpoint import save_checkpoint
from picotron_sft import run_sft

sft_model = PicotronDecoderModel(config).to(device)
sft_model.load_state_dict(
    load_file(sft_settings["checkpoints"]["base_checkpoint_path"], device=str(device))
)
sft_optimizer = AdamW(
    sft_model.parameters(),
    lr=sft_settings["optimizer"]["learning_rate_scheduler"]["learning_rate"],
)
sft_losses = run_sft(
    model=sft_model,
    dataset=smoltalk_batches(),
    batch_size=1,  # iterator already yields batches
    optimizer=sft_optimizer,
    num_steps=None,
    display_config=config,
    device=device,
)
sft_checkpoint = Path(sft_settings["checkpoints"]["output_checkpoint_path"])
save_checkpoint(sft_trainer.model, sft_trainer.optimizer, len(sft_losses), sft_checkpoint)
assert sft_checkpoint.exists()
print(f"SFT steps: {len(sft_losses)}; checkpoint: {sft_checkpoint}")


## 10. Generate after SFT and compare

The same model family and prompts are used after SFT. With a 1M-parameter model, compare formatting and continuation qualitatively, not benchmark quality.


In [ ]:
sft_inference_model = PicotronDecoderModel(config).to(device)
sft_inference_model.load_state_dict(
    load_file(str(sft_checkpoint), device=str(device))
)
sft_inference_model.eval()
sft_prompts = [
    "user: Explain photosynthesis simply.\nassistant:",
    "user: Write a short greeting for a new programmer.\nassistant:",
]
for prompt in sft_prompts:
    print(f"PROMPT: {prompt}\n{generate(sft_inference_model, prompt)}\n")
print("Pretraining outputs were printed in cell 6; compare them with these SFT outputs.")
print("GPU execution, network availability, runtime, and loss trends must be confirmed on Kaggle.")
